In [0]:
from pyspark.sql import functions as F

CATALOGO = "mvp"
ESQUEMA = "staging"

TABELA_SILVER = f"{CATALOGO}.{ESQUEMA}.silver_diabetes_clean"
TABELA_GOLD   = f"{CATALOGO}.{ESQUEMA}.gold_fato_diabetes"

df_silver = spark.table(TABELA_SILVER)

# Categorizações / Feature Engineering
df_gold = df_silver \
    .withColumn("faixa_etaria", 
        F.when(F.col("age") < 30, "Jovem (<30)")
         .when((F.col("age") >= 30) & (F.col("age") <= 59), "Adulto (30-59)")
         .otherwise("Idoso (60+)")
    ) \
    .withColumn("categoria_bmi", 
        F.when(F.col("bmi") < 25.0, "1. Normal")
         .when((F.col("bmi") >= 25.0) & (F.col("bmi") < 30.0), "2. Sobrepeso")
         .otherwise("3. Obesidade")
    ) \
    .withColumn("categoria_glicose", 
        F.when(F.col("fasting_blood_sugar") < 100.0, "1. Normal")
         .when((F.col("fasting_blood_sugar") >= 100.0) & (F.col("fasting_blood_sugar") <= 125.0), "2. Alterada")
         .otherwise("3. Elevada")
    )

# Persistência
df_gold.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(TABELA_GOLD)

print(f"✓ Camada Gold criada com sucesso! Linhas: {spark.table(TABELA_GOLD).count()}")

In [0]:
%sql
-- P1: Prevalência de Diabetes por Faixa Etária
SELECT 
  faixa_etaria, 
  COUNT(*) AS total_pacientes, 
  SUM(diabetes) AS casos_diabetes, 
  ROUND(AVG(diabetes) * 100, 2) AS pct_prevalencia
FROM mvp.staging.gold_fato_diabetes
GROUP BY faixa_etaria
ORDER BY pct_prevalencia DESC;

-- P2: Risco Combinado de IMC e Histórico Familiar
SELECT 
  categoria_bmi, 
  family_history_diabetes, 
  COUNT(*) AS total_pacientes, 
  ROUND(AVG(diabetes) * 100, 2) AS pct_prevalencia
FROM mvp.staging.gold_fato_diabetes
GROUP BY categoria_bmi, family_history_diabetes
ORDER BY categoria_bmi, family_history_diabetes;

-- P3: Impacto do Nível de Atividade Física na Glicemia
SELECT 
  physical_activity_level, 
  ROUND(AVG(fasting_blood_sugar), 2) AS media_glicose, 
  ROUND(AVG(diabetes) * 100, 2) AS pct_prevalencia
FROM mvp.staging.gold_fato_diabetes
GROUP BY physical_activity_level
ORDER BY media_glicose;

-- P4: Distribuição por Faixa Glicêmica
SELECT 
  categoria_glicose, 
  COUNT(*) AS total_pacientes, 
  ROUND(AVG(diabetes) * 100, 2) AS pct_prevalencia
FROM mvp.staging.gold_fato_diabetes
GROUP BY categoria_glicose
ORDER BY categoria_glicose;

In [0]:
from pyspark.sql import functions as F

TABELA_SILVER = "mvp.staging.silver_diabetes_clean"
TABELA_GOLD = "mvp.staging.gold_fato_diabetes"

df_silver = spark.table(TABELA_SILVER)

df_gold = df_silver \
    .withColumn(
        "faixa_etaria",
        F.when(F.col("age") < 30, "Jovem (<30)")
        .when((F.col("age") >= 30) & (F.col("age") <= 59), "Adulto (30-59)")
        .otherwise("Idoso (60+)")
    ) \
    .withColumn(
        "categoria_bmi",
        F.when(F.col("bmi") < 25.0, "1. Normal")
        .when((F.col("bmi") >= 25.0) & (F.col("bmi") < 30.0), "2. Sobrepeso")
        .otherwise("3. Obesidade")
    ) \
    .withColumn(
        "categoria_glicose",
        F.when(F.col("fasting_blood_sugar") < 100.0, "1. Normal")
        .when((F.col("fasting_blood_sugar") >= 100.0) & (F.col("fasting_blood_sugar") <= 125.0), "2. Alterada")
        .otherwise("3. Elevada")
    )

df_gold.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(TABELA_GOLD)

In [0]:
%sql
SELECT 
  COUNT(*) AS total_registros,
  SUM(diabetes) AS total_diabeticos,
  ROUND(AVG(diabetes) * 100, 2) AS pct_diabeticos,
  ROUND(AVG(fasting_blood_sugar), 2) AS media_glicose_geral,
  COUNT(DISTINCT categoria_glicose) AS qtd_categorias_glicose
FROM mvp.staging.gold_fato_diabetes;

In [0]:
%sql
SELECT
  family_history_diabetes,
  physical_activity_level,
  COUNT(*) AS total_pacientes,
  SUM(diabetes) AS casos_diabetes,
  ROUND((SUM(diabetes) / COUNT(*)) * 100, 2) AS pct_prevalencia
FROM mvp.staging.gold_fato_diabetes
WHERE physical_activity_level IS NOT NULL
GROUP BY family_history_diabetes, physical_activity_level
ORDER BY family_history_diabetes DESC, pct_prevalencia DESC;

In [0]:
%sql
SELECT
  family_history_diabetes,
  physical_activity_level,
  COUNT(*) AS total_pacientes,
  SUM(diabetes) AS casos_diabetes,
  ROUND((SUM(diabetes) / COUNT(*)) * 100, 2) AS pct_prevalencia
FROM mvp.staging.gold_fato_diabetes
WHERE physical_activity_level IS NOT NULL
GROUP BY family_history_diabetes, physical_activity_level
ORDER BY family_history_diabetes DESC, pct_prevalencia DESC;